In [7]:
import numpy as np

# Prahl, cm^-1/M : (HbO2, Hb) aux deux longueurs d'onde de l'oxymètre
eps = {660: (319.6, 3226.56), 940: (1214.0, 693.44)}
C_HB = 150 / 64500  # mol/L, comme dans bias_model.ipynb

def mua(lam, S):
    e_ox, e_de = eps[lam]
    return 2.303 * C_HB * (S * e_ox + (1 - S) * e_de)  # cm^-1

for S in (0.97, 0.85):
    print(S, {lam: round(mua(lam, S), 2) for lam in (660, 940)})

for PI in (0.005, 0.10):
    dd_um = PI / mua(940, 0.97) * 1e4  # cm -> µm
    print(f"PI {PI*100:.1f} % -> épaisseur pulsée {dd_um:.0f} µm")

0.97 {660: 2.18, 940: 6.42}
0.85 {660: 4.05, 940: 6.08}
PI 0.5 % -> épaisseur pulsée 8 µm
PI 10.0 % -> épaisseur pulsée 156 µm


In [8]:
D = 2.0   # mm, vaisseau_diam
L = 52.0  # mm, doigt_longueur - vaisseau_fond

def stroke_volume(dd_mm):
    dA = np.pi / 4 * ((D + dd_mm)**2 - D**2)  # mm²
    return dA * L                             # mm³ = µL

for PI in (0.005, 0.10):
    dd_mm = PI / mua(940, 0.97) * 10  # cm -> mm
    print(f"PI {PI*100:.1f} % -> ΔV = {stroke_volume(dd_mm):.1f} µL")

PI 0.5 % -> ΔV = 1.3 µL
PI 10.0 % -> ΔV = 26.4 µL


In [9]:
# diamètres intérieurs typiques (BD Plastipak), à mesurer au pied à coulisse
syringes = {"1 mL": 4.69, "3 mL": 8.66, "5 mL": 12.06}  # mm
V_max = 50.0                                            # µL, cible avec marge
V_min = stroke_volume(0.005 / mua(940, 0.97) * 10)      # µL, PI = 0,5 %

for name, d in syringes.items():
    A = np.pi / 4 * d**2  # mm²
    print(f"{name}: A = {A:.1f} mm², course max {V_max/A:.2f} mm, "
          f"course min {V_min/A*1000:.0f} µm")

1 mL: A = 17.3 mm², course max 2.89 mm, course min 74 µm
3 mL: A = 58.9 mm², course max 0.85 mm, course min 22 µm
5 mL: A = 114.2 mm², course max 0.44 mm, course min 11 µm


Choix : seringue 1 mL, car la course min (71 µm) reste grande devant le jeu vis-écrou à chaque inversion.

In [10]:
A = np.pi / 4 * 4.69**2          # mm², seringue 1 mL retenue
course_max = V_max / A           # mm
course_min = V_min / A           # mm

steps_rev = 200 * 16             # micropas par tour
bpm = 120                        # pire cas
t_up = 0.25 * 60 / bpm           # s : la poussée dure 1/4 du battement
v_peak = np.pi / 2 * course_max / t_up  # mm/s, vitesse de pointe

for lead in (8.0, 2.0, 0.8):     # mm/tour : vis T8 (pas 8), T8 (pas 2), tige M5
    res_um = lead / steps_rev * 1000
    rpm = v_peak / lead * 60
    f_step = v_peak / lead * steps_rev
    print(f"pas {lead} mm : {res_um:.2f} µm/micropas, "
          f"{course_min*1000/res_um:.0f} micropas pour le pouls min, "
          f"{rpm:.0f} tr/min, {f_step/1000:.0f} kHz")

pas 8.0 mm : 2.50 µm/micropas, 30 micropas pour le pouls min, 273 tr/min, 15 kHz
pas 2.0 mm : 0.62 µm/micropas, 118 micropas pour le pouls min, 1091 tr/min, 58 kHz
pas 0.8 mm : 0.25 µm/micropas, 295 micropas pour le pouls min, 2728 tr/min, 145 kHz


In [11]:
E = 100e3                        # Pa, silicone Shore 00-30 (ordre de grandeur)
dd_max = 0.10 / mua(940, 0.97) * 10   # mm, gonflement pour PI = 10 %
p_dil = (dd_max / D) * E / 1.5   # Pa

mu = 1e-3                        # Pa·s, viscosité de l'eau
L_tube, d_tube = 0.30, 1.5e-3    # m, tuyau de 30 cm, Ø intérieur 1,5 mm
Q = v_peak * 1e-3 * A * 1e-6     # m³/s, débit de pointe
dp_tube = 128 * mu * L_tube * Q / (np.pi * d_tube**4)  # Pa

F_fric = 3.0                     # N, joint de seringue 1 mL (à mesurer)
F = (p_dil + dp_tube) * A * 1e-6 + F_fric  # N (A passe de mm² en m²)

eta, lead_m = 0.4, 8e-3          # rendement prudent, pas 8 mm
T = F * lead_m / (2 * np.pi * eta)  # N·m
print(f"p = {p_dil:.0f} Pa ({p_dil/133.3:.0f} mmHg), tuyau {dp_tube:.0f} Pa, "
      f"F = {F:.2f} N, couple {T*1000:.1f} mN·m, marge x{0.3/T:.0f}")


p = 5193 Pa (39 mmHg), tuyau 1517 Pa, F = 3.12 N, couple 9.9 mN·m, marge x30


tuyau de diamètre intérieur ≥ 1,5 mm, le plus court possible (≤ 30 cm). C’est la même logique que pour la seringue : on dimensionne pour que les pertes restent petites devant l’effet utile.

## Summary: pulse mechanism sizing

Requirements: 40–120 bpm, IR perfusion index 0.5–10 %.

| Quantity | Result | Decision |
|---|---|---|
| Channel dilation | 8–156 µm (upper bound, straight light path) | |
| Stroke volume | 1.3–26 µL | design for 50 µL |
| Syringe | stroke 74 µm – 2.9 mm | 1 mL (min stroke ≫ backlash) |
| Lead screw | 2.5 µm/microstep, 273 rpm, 14.5 kHz at 120 bpm | T8, lead 8 mm, anti-backlash nut |
| Force / torque | 3.1 N / 10 mN·m (plunger friction dominates) | NEMA 17, margin ×30 |
| Tubing | 1.5 kPa loss (7.7 kPa at 1 mm ID) | ID ≥ 1.5 mm, ≤ 30 cm |

Syringe used: Henke Sass Wolf NORM-JECT 1 mL, luer slip (inner diameter 4.69 mm, outer diameter 6.40 mm, manufacturer data). Two-part syringe (no rubber seal): stick-slip possible.

Driver: TMC2209 (silent mode, less vibration near the oximeter; current can be set low).

To check on the real parts: plunger friction (kitchen scale), silicone stiffness, tubing compliance and air bubbles.